# 🔬 Phase 1: Data Preparation & Synthetic Data Generation
**Đề tài:** Nghiên cứu xây dựng hệ thống trích xuất và cấu trúc hóa thông tin tự động từ văn bản hành chính tiếng Việt sử dụng LLMs và OCR

**Notebook này thực hiện:**
1. ⚙️ Cài đặt môi trường
2. 🔴 Trích xuất con dấu đỏ từ 150 PDF test
3. 🔴 Tạo 200+ synthetic stamps giống thật
4. 📄 Chuyển 2000 docx → Image + Overlay stamps → Training pairs cho GAN
5. 🧠 Tạo instruction dataset cho LLM fine-tuning (từ 2000 docx)

> **Yêu cầu:** Upload folder `data/` lên Google Drive trước khi chạy

## ⚙️ Cell 1: Cài đặt packages

In [ ]:
!pip install -q python-docx Pillow opencv-python-headless numpy pandas PyMuPDF tqdm

## 📂 Cell 2: Mount Google Drive & Cấu hình đường dẫn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import cv2
import json
import random
import math
import re
import zipfile
import xml.etree.ElementTree as ET
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from tqdm import tqdm

# ====== CẤU HÌNH ĐƯỜNG DẪN ======
BASE_DIR = "/content/drive/MyDrive/OCR-LLM_Research"

DATA_DIR = os.path.join(BASE_DIR, "data")
RAW_DOCX_DIR = os.path.join(DATA_DIR, "raw_word_files")
TEST_PDF_DIR = os.path.join(DATA_DIR, "test")
STAMPS_EXTRACTED_DIR = os.path.join(DATA_DIR, "stamps/extracted")
STAMPS_SYNTHETIC_DIR = os.path.join(DATA_DIR, "stamps/synthetic")
CLEAN_IMAGES_DIR = os.path.join(DATA_DIR, "processed/clean_images")
STAMPED_IMAGES_DIR = os.path.join(DATA_DIR, "processed/stamped_images")
LLM_TRAINING_DIR = os.path.join(DATA_DIR, "llm_training")

for d in [STAMPS_EXTRACTED_DIR, STAMPS_SYNTHETIC_DIR,
          CLEAN_IMAGES_DIR, STAMPED_IMAGES_DIR, LLM_TRAINING_DIR]:
    os.makedirs(d, exist_ok=True)

# Kiểm tra dữ liệu
if os.path.exists(RAW_DOCX_DIR):
    categories = os.listdir(RAW_DOCX_DIR)
    for cat in categories:
        cat_dir = os.path.join(RAW_DOCX_DIR, cat)
        if os.path.isdir(cat_dir):
            count = len([f for f in os.listdir(cat_dir) if f.endswith('.docx')])
            print(f"  📁 {cat}: {count} files")
else:
    print("⚠️ Không tìm thấy raw_word_files! Hãy upload data lên Google Drive.")

if os.path.exists(TEST_PDF_DIR):
    pdf_count = len([f for f in os.listdir(TEST_PDF_DIR) if f.endswith('.pdf')])
    print(f"  📁 Test PDFs: {pdf_count} files")
else:
    print("⚠️ Không tìm thấy test PDFs!")

print(f"\n✅ BASE_DIR: {BASE_DIR}")

---
## 🔴 Cell 3: Trích xuất con dấu đỏ từ 150 PDF test
Sử dụng **HSV color segmentation** để tự động phát hiện và crop vùng con dấu đỏ.

In [ ]:
import fitz  # PyMuPDF

def extract_stamps_from_pdf(pdf_path, output_dir, min_area=800, max_stamps=3):
    """
    Trích xuất con dấu đỏ từ PDF bằng HSV color segmentation.
    Pipeline: Render page → HSV → Red mask → Contours → Crop stamp
    """
    doc = fitz.open(pdf_path)
    pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
    extracted_paths = []

    for page_idx in range(min(len(doc), 5)):
        page = doc[page_idx]
        pix = page.get_pixmap(dpi=200)
        img_array = np.frombuffer(pix.samples, dtype=np.uint8).reshape(
            pix.height, pix.width, pix.n
        )
        if pix.n == 4:
            img_bgr = cv2.cvtColor(img_array, cv2.COLOR_RGBA2BGR)
        else:
            img_bgr = cv2.cvtColor(img_array, cv2.COLOR_RGB2BGR)

        hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

        # Mask cho vùng đỏ (dấu đỏ nằm ở 2 vùng Hue)
        mask1 = cv2.inRange(hsv, np.array([0, 60, 60]), np.array([10, 255, 255]))
        mask2 = cv2.inRange(hsv, np.array([160, 60, 60]), np.array([180, 255, 255]))
        red_mask = cv2.bitwise_or(mask1, mask2)

        # Morphological cleaning
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
        red_mask = cv2.morphologyEx(red_mask, cv2.MORPH_CLOSE, kernel, iterations=3)
        red_mask = cv2.morphologyEx(red_mask, cv2.MORPH_OPEN, kernel, iterations=1)

        contours, _ = cv2.findContours(red_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        valid_contours = sorted(
            [c for c in contours if cv2.contourArea(c) > min_area],
            key=cv2.contourArea, reverse=True
        )

        for i, contour in enumerate(valid_contours[:max_stamps]):
            x, y, w, h = cv2.boundingRect(contour)
            aspect_ratio = w / max(h, 1)
            if aspect_ratio < 0.3 or aspect_ratio > 3.0 or w < 60 or h < 60:
                continue

            pad = 15
            x1, y1 = max(0, x - pad), max(0, y - pad)
            x2 = min(img_bgr.shape[1], x + w + pad)
            y2 = min(img_bgr.shape[0], y + h + pad)

            stamp_crop = img_bgr[y1:y2, x1:x2]
            stamp_mask_crop = red_mask[y1:y2, x1:x2]

            # RGBA với alpha = red mask
            stamp_rgba = cv2.cvtColor(stamp_crop, cv2.COLOR_BGR2BGRA)
            stamp_rgba[:, :, 3] = stamp_mask_crop

            stamp_filename = f"{pdf_name}_p{page_idx}_stamp{i}.png"
            stamp_path = os.path.join(output_dir, stamp_filename)
            cv2.imwrite(stamp_path, stamp_rgba)
            extracted_paths.append(stamp_path)

    doc.close()
    return extracted_paths


# ====== CHẠY TRÍCH XUẤT ======
print("🔍 Đang trích xuất con dấu từ 150 PDFs...")
pdf_files = sorted([f for f in os.listdir(TEST_PDF_DIR) if f.endswith('.pdf')])
all_stamps = []

for i, pdf_file in enumerate(tqdm(pdf_files, desc="Extracting stamps")):
    try:
        stamps = extract_stamps_from_pdf(
            os.path.join(TEST_PDF_DIR, pdf_file), STAMPS_EXTRACTED_DIR
        )
        all_stamps.extend(stamps)
    except Exception as e:
        pass  # Skip problematic PDFs

print(f"\n✅ Trích xuất được {len(all_stamps)} stamps từ {len(pdf_files)} PDFs")
print(f"   Lưu tại: {STAMPS_EXTRACTED_DIR}")

### 👁️ Xem thử stamps đã trích xuất

In [ ]:
import matplotlib.pyplot as plt

stamp_files = [f for f in os.listdir(STAMPS_EXTRACTED_DIR) if f.endswith('.png')][:12]
if stamp_files:
    fig, axes = plt.subplots(2, min(6, len(stamp_files)), figsize=(20, 7))
    axes = axes.flatten() if len(stamp_files) > 1 else [axes]
    for ax, sf in zip(axes, stamp_files):
        img = Image.open(os.path.join(STAMPS_EXTRACTED_DIR, sf))
        ax.imshow(img)
        ax.set_title(sf[:20], fontsize=8)
        ax.axis('off')
    for ax in axes[len(stamp_files):]:
        ax.axis('off')
    plt.suptitle(f'Stamps trích xuất từ PDF ({len(stamp_files)} mẫu)', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('Chưa có stamps nào được trích xuất.')

---
## 🔴 Cell 4: Tạo 200 Synthetic Stamps giống thật
Tạo con dấu giả lập bằng Python với:
- Viền tròn/oval đỏ (2 viền)
- Ngôi sao 5 cánh ở giữa
- Tên cơ quan bao quanh
- Random rotation, blur, opacity

In [ ]:
def _draw_star(draw, cx, cy, size, color, opacity):
    """Vẽ ngôi sao 5 cánh."""
    points = []
    for i in range(10):
        angle = math.radians(i * 36 - 90)
        r = size if i % 2 == 0 else size * 0.4
        points.append((cx + r * math.cos(angle), cy + r * math.sin(angle)))
    draw.polygon(points, fill=(*color, opacity))


def create_synthetic_stamp(output_path, org_name="CƠ QUAN", sub_text="",
                           stamp_type="circle", size=250, color=(200, 30, 30),
                           has_star=True, rotation_range=(-15, 15),
                           blur_amount=0.5, opacity=200):
    """Tạo con dấu giả lập giống thật."""
    canvas_size = int(size * 1.5)
    img = Image.new('RGBA', (canvas_size, canvas_size), (0, 0, 0, 0))
    draw = ImageDraw.Draw(img)

    cx, cy = canvas_size // 2, canvas_size // 2
    radius = size // 2
    border_width = max(3, size // 35)

    rx = radius
    ry = int(radius * 0.78) if stamp_type == 'oval' else radius

    # Viền ngoài
    draw.ellipse([cx-rx, cy-ry, cx+rx, cy+ry],
                 outline=(*color, opacity), width=border_width)
    # Viền trong
    gap = max(4, size // 25)
    draw.ellipse([cx-rx+gap, cy-ry+gap, cx+rx-gap, cy+ry-gap],
                 outline=(*color, opacity), width=max(1, border_width//2))

    # Ngôi sao
    if has_star:
        _draw_star(draw, cx, cy, size // 6, color, opacity)

    # Font
    font_paths = ["/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
                  "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"]
    font = None
    for fp in font_paths:
        if os.path.exists(fp):
            font = ImageFont.truetype(fp, max(10, size // 12))
            break
    if font is None:
        font = ImageFont.load_default()

    # Text cong phía trên
    text_radius = radius - gap - max(10, size // 12)
    chars = list(org_name.upper())
    if chars:
        angle_span = min(150, len(chars) * 12)
        start_angle = 270 - angle_span // 2
        for j, ch in enumerate(chars):
            angle = math.radians(start_angle + j * (angle_span / max(len(chars)-1, 1)))
            x = cx + text_radius * math.cos(angle)
            y = cy + text_radius * math.sin(angle)
            draw.text((x-5, y-5), ch, font=font, fill=(*color, opacity))

    # Text phía dưới (sub_text)
    if sub_text:
        small_font = None
        for fp in font_paths:
            if os.path.exists(fp):
                small_font = ImageFont.truetype(fp, max(8, size // 16))
                break
        if small_font is None:
            small_font = ImageFont.load_default()
        text_w = small_font.getlength(sub_text) if hasattr(small_font, 'getlength') else len(sub_text) * 6
        draw.text((cx - text_w/2, cy + ry//2), sub_text,
                  font=small_font, fill=(*color, opacity))

    # Xoay ngẫu nhiên
    rotation = random.uniform(*rotation_range)
    img = img.rotate(rotation, resample=Image.BICUBIC, expand=False)

    # Blur nhẹ
    if blur_amount > 0:
        img = img.filter(ImageFilter.GaussianBlur(radius=blur_amount))

    # Crop
    bbox = img.getbbox()
    if bbox:
        img = img.crop(bbox)

    img.save(output_path, 'PNG')
    return img


# ====== TẠO 200 SYNTHETIC STAMPS ======
ORG_NAMES = [
    "ỦY BAN NHÂN DÂN", "BỘ GIÁO DỤC VÀ ĐÀO TẠO", "BỘ CÔNG AN",
    "BỘ TÀI CHÍNH", "BỘ Y TẾ", "BỘ NGOẠI GIAO", "BỘ TƯ PHÁP",
    "TRƯỜNG ĐẠI HỌC BÁCH KHOA", "TRƯỜNG ĐẠI HỌC CÔNG NGHỆ",
    "TRƯỜNG ĐẠI HỌC KHOA HỌC TỰ NHIÊN", "TRƯỜNG ĐẠI HỌC SƯ PHẠM",
    "TRƯỜNG ĐẠI HỌC KINH TẾ", "TRƯỜNG ĐẠI HỌC LUẬT",
    "SỞ GIÁO DỤC VÀ ĐÀO TẠO", "SỞ TÀI CHÍNH", "SỞ CÔNG THƯƠNG",
    "SỞ KẾ HOẠCH VÀ ĐẦU TƯ", "SỞ TÀI NGUYÊN VÀ MÔI TRƯỜNG",
    "CHI CỤC THUẾ", "CỤC THUẾ", "KHO BẠC NHÀ NƯỚC",
    "NGÂN HÀNG NHÀ NƯỚC", "VIỆN KIỂM SÁT NHÂN DÂN",
    "TÒA ÁN NHÂN DÂN", "HỘI ĐỒNG NHÂN DÂN",
    "TỔNG CỤC HẢI QUAN", "CỤC QUẢN LÝ THỊ TRƯỜNG",
    "UBND QUẬN", "UBND PHƯỜNG", "UBND HUYỆN", "UBND XÃ",
    "UBND THÀNH PHỐ", "UBND TỈNH",
    "CÔNG TY TNHH", "CÔNG TY CỔ PHẦN", "TỔNG CÔNG TY",
    "BAN QUẢN LÝ DỰ ÁN", "TRUNG TÂM Y TẾ", "BỆNH VIỆN ĐA KHOA",
]

SUB_TEXTS = [
    "TP. HỒ CHÍ MINH", "HÀ NỘI", "ĐÀ NẴNG", "CẦN THƠ",
    "BÌNH DƯƠNG", "ĐỒNG NAI", "HẢI PHÒNG", "NGHỆ AN",
    "", "", "",  # Một số không có sub_text
]

print("🔴 Đang tạo 200 synthetic stamps...")
for i in tqdm(range(200), desc="Generating stamps"):
    org = random.choice(ORG_NAMES)
    sub = random.choice(SUB_TEXTS)
    create_synthetic_stamp(
        output_path=os.path.join(STAMPS_SYNTHETIC_DIR, f"stamp_syn_{i:04d}.png"),
        org_name=org, sub_text=sub,
        stamp_type=random.choice(["circle", "circle", "oval"]),
        size=random.randint(180, 320),
        color=(random.randint(170, 220), random.randint(10, 50), random.randint(10, 50)),
        has_star=random.random() > 0.1,
        rotation_range=(-20, 20),
        blur_amount=random.uniform(0.3, 1.2),
        opacity=random.randint(160, 240)
    )

syn_count = len([f for f in os.listdir(STAMPS_SYNTHETIC_DIR) if f.endswith('.png')])
print(f"\n✅ Tạo xong {syn_count} synthetic stamps tại {STAMPS_SYNTHETIC_DIR}")

### 👁️ Xem thử synthetic stamps

In [ ]:
syn_files = sorted([f for f in os.listdir(STAMPS_SYNTHETIC_DIR) if f.endswith('.png')])[:12]
fig, axes = plt.subplots(2, 6, figsize=(20, 7))
for ax, sf in zip(axes.flatten(), syn_files):
    img = Image.open(os.path.join(STAMPS_SYNTHETIC_DIR, sf))
    ax.imshow(img)
    ax.axis('off')
plt.suptitle('Synthetic Stamps (mẫu)', fontsize=14)
plt.tight_layout()
plt.show()

---
## 📄 Cell 5: Chuyển DOCX → Clean Image + Stamped Image
Tạo training pairs cho Pix2Pix GAN: `{ảnh sạch}` ↔ `{ảnh có dấu}`

In [ ]:
def docx_to_text(docx_path):
    """Đọc text từ docx bằng zipfile (không cần python-docx)."""
    try:
        with zipfile.ZipFile(docx_path, 'r') as z:
            xml_content = z.read('word/document.xml')
        tree = ET.fromstring(xml_content)
        ns = 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'
        paragraphs = []
        for p in tree.findall(f'.//{{{ns}}}p'):
            texts = [t.text for t in p.findall(f'.//{{{ns}}}t') if t.text]
            line = ''.join(texts).strip()
            if line:
                paragraphs.append(line)
        return '\n'.join(paragraphs)
    except Exception as e:
        return ""


def text_to_image(text, output_path, width=2480, min_height=3508, font_size=30, margin=100):
    """Render text thành ảnh giống trang A4 scan (300dpi)."""
    font_paths = ["/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
                  "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf"]
    font = None
    for fp in font_paths:
        if os.path.exists(fp):
            font = ImageFont.truetype(fp, font_size)
            break
    if font is None:
        font = ImageFont.load_default()

    # Word wrap
    lines = []
    for paragraph in text.split('\n'):
        if not paragraph.strip():
            lines.append('')
            continue
        words = paragraph.split()
        current = ''
        for word in words:
            test = f"{current} {word}".strip()
            bbox = font.getbbox(test)
            if bbox[2] > width - 2 * margin:
                lines.append(current)
                current = word
            else:
                current = test
        if current:
            lines.append(current)

    line_height = int(font_size * 1.5)
    height = max(min_height, len(lines) * line_height + 2 * margin)
    img = Image.new('RGB', (width, height), (255, 255, 255))
    draw = ImageDraw.Draw(img)

    y = margin
    for line in lines:
        draw.text((margin, y), line, font=font, fill=(0, 0, 0))
        y += line_height

    img.save(output_path, 'PNG', optimize=True)
    return img


def overlay_stamp(clean_path, stamp_path, output_path):
    """Overlay stamp lên ảnh sạch → tạo ảnh noisy cho GAN training."""
    clean = Image.open(clean_path).convert('RGBA')
    stamp = Image.open(stamp_path).convert('RGBA')

    # Resize stamp (15-22% chiều rộng trang)
    max_w = int(clean.width * random.uniform(0.13, 0.22))
    scale = max_w / max(stamp.width, 1)
    stamp = stamp.resize((int(stamp.width * scale), int(stamp.height * scale)), Image.LANCZOS)

    # Vị trí: thường ở phần dưới bên phải (như dấu thật)
    positions = ['bottom_right', 'bottom_center', 'random']
    pos = random.choice(positions)
    if pos == 'bottom_right':
        x = clean.width - stamp.width - random.randint(50, 200)
        y = clean.height - stamp.height - random.randint(100, 400)
    elif pos == 'bottom_center':
        x = (clean.width - stamp.width) // 2 + random.randint(-100, 100)
        y = clean.height - stamp.height - random.randint(100, 400)
    else:
        x = random.randint(clean.width // 3, clean.width - stamp.width - 50)
        y = random.randint(clean.height // 2, clean.height - stamp.height - 50)

    x = max(0, min(x, clean.width - stamp.width))
    y = max(0, min(y, clean.height - stamp.height))

    clean.paste(stamp, (x, y), stamp)
    clean.convert('RGB').save(output_path, 'PNG', optimize=True)


# ====== TẠO TRAINING PAIRS ======
# Gộp tất cả stamps (extracted + synthetic)
all_stamp_files = []
for d in [STAMPS_EXTRACTED_DIR, STAMPS_SYNTHETIC_DIR]:
    if os.path.exists(d):
        all_stamp_files.extend([
            os.path.join(d, f) for f in os.listdir(d) if f.endswith('.png')
        ])
print(f"🔴 Tổng stamps: {len(all_stamp_files)}")

# Lấy tất cả docx files
docx_files = []
for category in os.listdir(RAW_DOCX_DIR):
    cat_dir = os.path.join(RAW_DOCX_DIR, category)
    if os.path.isdir(cat_dir):
        for f in os.listdir(cat_dir):
            if f.endswith('.docx'):
                docx_files.append((os.path.join(cat_dir, f), category))

PAIR_LIMIT = 500  # Bắt đầu với 500 cặp, tăng lên nếu cần
docx_subset = docx_files[:PAIR_LIMIT]

print(f"📝 Đang tạo {len(docx_subset)} training pairs (docx → image + stamp)...")
pairs_created = 0

for docx_path, category in tqdm(docx_subset, desc="Creating pairs"):
    filename = os.path.splitext(os.path.basename(docx_path))[0]
    text = docx_to_text(docx_path)
    if len(text) < 50:
        continue

    clean_path = os.path.join(CLEAN_IMAGES_DIR, f"{category}_{filename}.png")
    stamped_path = os.path.join(STAMPED_IMAGES_DIR, f"{category}_{filename}.png")

    try:
        text_to_image(text, clean_path)
        stamp_file = random.choice(all_stamp_files)
        overlay_stamp(clean_path, stamp_file, stamped_path)
        pairs_created += 1
    except Exception as e:
        pass

print(f"\n✅ Tạo xong {pairs_created} training pairs")
print(f"   Clean: {CLEAN_IMAGES_DIR}")
print(f"   Stamped: {STAMPED_IMAGES_DIR}")

### 👁️ Xem thử training pairs (Clean vs Stamped)

In [ ]:
clean_files = sorted([f for f in os.listdir(CLEAN_IMAGES_DIR) if f.endswith('.png')])[:4]
fig, axes = plt.subplots(2, 4, figsize=(20, 14))
for idx, f in enumerate(clean_files):
    clean_img = Image.open(os.path.join(CLEAN_IMAGES_DIR, f))
    stamped_path = os.path.join(STAMPED_IMAGES_DIR, f)
    # Crop phần dưới (có stamp) để zoom
    h = clean_img.height
    crop_box = (0, h//2, clean_img.width, h)
    axes[0][idx].imshow(clean_img.crop(crop_box))
    axes[0][idx].set_title(f'Clean (bottom half)', fontsize=9)
    axes[0][idx].axis('off')
    if os.path.exists(stamped_path):
        stamped_img = Image.open(stamped_path)
        axes[1][idx].imshow(stamped_img.crop(crop_box))
        axes[1][idx].set_title(f'With Stamp', fontsize=9)
    axes[1][idx].axis('off')
plt.suptitle('Training Pairs: Clean vs Stamped (bottom half)', fontsize=14)
plt.tight_layout()
plt.show()

---
## 🧠 Cell 6: Tạo LLM Training Dataset (Instruction-Following)
Parse 2000 docx → trích xuất metadata → tạo Alpaca-style dataset cho fine-tuning Qwen-7B

In [ ]:
def extract_metadata(docx_path, category):
    """Trích xuất metadata từ docx bằng regex."""
    text = docx_to_text(docx_path)
    if not text or len(text) < 30:
        return None, None

    cat_map = {'CV': 'Công văn', 'HD': 'Hợp đồng', 'QD': 'Quy định',
               'TT': 'Tờ trình', 'K': 'Khác'}

    metadata = {
        "loai_van_ban": cat_map.get(category, "Khác"),
        "so_hieu": "", "ngay_ban_hanh": "",
        "trich_yeu": "", "nguoi_ky": "", "co_quan_ban_hanh": ""
    }

    # Số hiệu
    m = re.search(r'[Ss]ố[:\s]+(\d+[\/-][A-ZĐa-zđ\d\/-]+)', text)
    if m: metadata['so_hieu'] = m.group(1).strip()

    # Ngày
    m = re.search(r'[Nn]gày\s+(\d{1,2})\s+tháng\s+(\d{1,2})\s+năm\s+(\d{4})', text)
    if m:
        d, mo, y = m.groups()
        metadata['ngay_ban_hanh'] = f"{d.zfill(2)}/{mo.zfill(2)}/{y}"
    else:
        m = re.search(r'(\d{1,2})[\/-](\d{1,2})[\/-](\d{4})', text)
        if m:
            d, mo, y = m.groups()
            metadata['ngay_ban_hanh'] = f"{d.zfill(2)}/{mo.zfill(2)}/{y}"

    # Trích yếu
    m = re.search(r'[Vv]\/[Vv][:\s]+(.+?)(?:\n|$)', text)
    if m: metadata['trich_yeu'] = m.group(1).strip()[:200]

    # Người ký (heuristic: tên người ở cuối văn bản)
    lines = text.strip().split('\n')
    for line in reversed(lines[-10:]):
        line = line.strip()
        if line and len(line) < 50:
            words = line.split()
            if 2 <= len(words) <= 5 and all(w[0].isupper() for w in words if w):
                if not any(kw in line.lower() for kw in ['điều', 'khoản', 'mục', 'số']):
                    metadata['nguoi_ky'] = line
                    break

    return text, metadata


# ====== TẠO DATASET ======
CLASSIFICATION_INSTRUCTION = (
    "Bạn là chuyên gia phân loại văn bản hành chính Việt Nam. "
    "Hãy phân loại văn bản sau vào một trong các loại: "
    "Công văn, Hợp đồng, Quy định, Tờ trình, Khác. "
    "Chỉ trả lời tên loại văn bản, không giải thích thêm."
)

EXTRACTION_INSTRUCTION = (
    "Bạn là chuyên gia trích xuất thông tin từ văn bản hành chính Việt Nam. "
    "Hãy đọc văn bản sau và trích xuất các thông tin theo định dạng JSON:\n"
    "{\n"
    '  \"loai_van_ban\": \"<Công văn|Hợp đồng|Quy định|Tờ trình|Khác>\",\n'
    '  \"so_hieu\": \"<số hiệu văn bản>\",\n'
    '  \"ngay_ban_hanh\": \"<DD/MM/YYYY>\",\n'
    '  \"co_quan_ban_hanh\": \"<tên cơ quan>\",\n'
    '  \"trich_yeu\": \"<trích yếu nội dung>\",\n'
    '  \"nguoi_ky\": \"<họ tên người ký>\"\n'
    "}\n"
    "Nếu không tìm thấy thông tin, để trống (\"\")."
)

cat_map = {'CV': 'Công văn', 'HD': 'Hợp đồng', 'QD': 'Quy định',
           'TT': 'Tờ trình', 'K': 'Khác'}

dataset = []
print("📊 Đang tạo LLM instruction dataset từ 2000 docx files...")

for docx_path, category in tqdm(docx_files, desc="Building dataset"):
    text, metadata = extract_metadata(docx_path, category)
    if not text or len(text) < 30:
        continue

    # Truncate nếu quá dài
    input_text = text[:3000] if len(text) > 3000 else text

    # Task 1: Classification
    dataset.append({
        "instruction": CLASSIFICATION_INSTRUCTION,
        "input": input_text,
        "output": cat_map.get(category, "Khác")
    })

    # Task 2: Extraction
    if metadata:
        dataset.append({
            "instruction": EXTRACTION_INSTRUCTION,
            "input": input_text,
            "output": json.dumps(metadata, ensure_ascii=False, indent=2)
        })

# Shuffle & Split 80/10/10
random.shuffle(dataset)
n = len(dataset)
splits = {
    'train': dataset[:int(n*0.8)],
    'val': dataset[int(n*0.8):int(n*0.9)],
    'test': dataset[int(n*0.9):]
}

for name, data in splits.items():
    path = os.path.join(LLM_TRAINING_DIR, f"{name}.json")
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"  💾 {name}: {len(data)} samples → {path}")

print(f"\n✅ Tổng: {len(dataset)} instruction samples")

### 📊 Thống kê dataset

In [ ]:
# Xem mẫu dataset
print("=" * 60)
print("📊 THỐNG KÊ DỮ LIỆU ĐÃ TẠO")
print("=" * 60)

ext_count = len([f for f in os.listdir(STAMPS_EXTRACTED_DIR) if f.endswith('.png')])
syn_count = len([f for f in os.listdir(STAMPS_SYNTHETIC_DIR) if f.endswith('.png')])
clean_count = len([f for f in os.listdir(CLEAN_IMAGES_DIR) if f.endswith('.png')])
stamped_count = len([f for f in os.listdir(STAMPED_IMAGES_DIR) if f.endswith('.png')])

print(f"🔴 Stamps extracted:  {ext_count}")
print(f"🔴 Stamps synthetic:  {syn_count}")
print(f"📄 Clean images:      {clean_count}")
print(f"📄 Stamped images:    {stamped_count}")

for split in ['train', 'val', 'test']:
    p = os.path.join(LLM_TRAINING_DIR, f"{split}.json")
    if os.path.exists(p):
        with open(p, 'r', encoding='utf-8') as f:
            d = json.load(f)
        print(f"🧠 LLM {split}:          {len(d)} samples")

# Xem 1 mẫu
print("\n" + "=" * 60)
print("📝 MẪU DATASET (extraction task):")
print("=" * 60)
sample = [s for s in dataset if '{' in s.get('output', '')][:1]
if sample:
    print(f"Input (first 200 chars): {sample[0]['input'][:200]}...")
    print(f"Output: {sample[0]['output']}")

---
## ✅ Phase 1 hoàn tất!
**Tiếp theo:** Chạy `Phase2_Stamp_Removal_GAN.ipynb` để train GAN xóa dấu.

**Dữ liệu đã tạo:**
- `data/stamps/extracted/` - Stamps trích xuất từ PDF
- `data/stamps/synthetic/` - 200 synthetic stamps
- `data/processed/clean_images/` - Ảnh sạch
- `data/processed/stamped_images/` - Ảnh có overlay stamp
- `data/llm_training/` - train.json, val.json, test.json